# Satellite-Derived Chlorophyll Time Series for Lakes: MODIS 250m 8-Day Composite

## Overview

This notebook extracts chlorophyll-a concentration estimates using a NIR/Red ratio algorithm from MODIS Aqua 8-day composite data at 250m resolution for specified lake locations.

The MYD09Q1 product provides Aqua MODIS surface reflectance for bands 1 and 2 at 250m resolution, composited over 8-day periods and corrected for atmospheric conditions such as gases, aerosols, and Rayleigh scattering.

- **Purpose**: Generate time series of chlorophyll indices from high-resolution satellite imagery
- **Study Areas**: Detroit Lake and Upper Klamath Lake
- **Satellite Sensor**: MODIS-Aqua only (no Terra equivalent at 250m)
- **Algorithm**: Two-band NIR/Red ratio with multiple indices
- **Resolution**: 250m (4x higher resolution than 500m products)
- **Temporal**: 8-day composites (reduced temporal but increased spatial resolution)
- **Output**: CSV files with date-stamped chlorophyll index values

## Algorithm Background

The NIR/Red algorithm is specifically designed for turbid, productive waters (Case 2) where traditional blue-green algorithms fail due to interference from suspended sediments and colored dissolved organic matter (CDOM). The algorithm exploits:

- **Band 1 (Red, 620-670nm)**: Chlorophyll absorption maximum
- **Band 2 (NIR, 841-876nm)**: Baseline correction, minimal CDOM/sediment effects

## Advantages of 250m Resolution

- **Higher spatial detail**: 4x better resolution than 500m products
- **Reduced mixed pixels**: Better separation at shorelines
- **Spatial heterogeneity**: Ability to detect within-lake variations
- **Cloud-free composites**: 8-day compositing reduces cloud contamination

## Trade-offs

- **Temporal resolution**: 8-day composites vs daily observations
- **Single satellite**: Aqua only (no Terra comparison)
- **Limited spectral**: Only 2 bands (Red + NIR)

**Key References:**
- Gitelson et al. (2008): Simple semi-analytical model for remote estimation of chlorophyll-a in turbid waters
- Moses et al. (2009): NIR/Red algorithms for turbid waters
- Binding et al. (2012): MODIS-derived algal turbidity in lakes

In [1]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

In [2]:
# -----------------------------
# User parameters
# -----------------------------

lakes = [
    dict(
        name='Detroit',
        lon=-122.184, lat=44.711,
        aqua_export='Detroit_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m',
        # Set to None to skip Chl computation (export indices only)
        a=None, b=None
    ),
    dict(
        name='UpperKlamath',
        lon=-121.900, lat=42.400,
        aqua_export='UKL_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m',
        a=None, b=None
    ),
]

start_date = '2011-01-01'
end_date   = '2025-12-31'

# If a lake has a/b=None, Chl-a won't be computed (only indices exported)
DEFAULT_A = None
DEFAULT_B = None

# ROI & masking (adjusted for higher resolution)
ROI_RADIUS_M         = 500     # reduced from 1000m due to higher resolution
SHORELINE_BUFFER_M   = 250     # reduced from 500m for 250m pixels
WATER_OCC_THRESHOLD  = 75      # JRC occurrence threshold (0-100)
GSW_DATASET_ID       = 'JRC/GSW1_4/GlobalSurfaceWater'  # use '...1_3...' if needed

# MODIS collection and bands (250m 8-day composite)
AQUA_COL_ID  = 'MODIS/061/MYD09Q1'  # Aqua only - no Terra equivalent
B_RED        = 'sur_refl_b01'        # Band 1: ~645 nm (Red, 250 m)
B_NIR        = 'sur_refl_b02'        # Band 2: ~859 nm (NIR, 250 m)
SR_SCALE     = 1e-4                  # scale factor

In [ ]:
def mask_myd09q1(img):
    """
    MYD09Q1 QA-based mask using QA band.
    Simplified approach - start with minimal masking to ensure data availability.
    """
    # Start with minimal QA masking to avoid over-filtering
    # You can make this more restrictive once we confirm data availability
    return img  # No masking initially to debug data availability

In [4]:
# --------------------------------------
# Main loop with client-side CSV export
# --------------------------------------

for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(ROI_RADIUS_M)  # 500m radius for 250m resolution

    a = lake.get('a', DEFAULT_A)
    b = lake.get('b', DEFAULT_B)

    # Only Aqua data available for MYD09Q1
    aqua_fc = imagecollection_to_features(AQUA_COL_ID, roi, 'Aqua_250m_8day', a, b)

    # Availability
    print(f"{lake['name']} Aqua 250m 8-day features =", aqua_fc.size().getInfo())

    # ---------- Client-side export ----------
    # Aqua only
    aqua_rows = aqua_fc.getInfo()['features']
    aqua_records = [f['properties'] for f in aqua_rows]
    aqua_df = pd.DataFrame.from_records(aqua_records)
    aqua_df = aqua_df.sort_values('datetime') if 'datetime' in aqua_df.columns else aqua_df
    aqua_df.to_csv(lake['aqua_export'] + '.csv', index=False)
    print(f"Exported: {lake['aqua_export']}.csv")
    
    print(f"Completed processing for {lake['name']} Lake\n")

print("Processing complete!")
print("\nOutput files contain the following indices at 250m resolution:")
print("- nir_red_ratio: NIR/Red ratio (primary algorithm for turbid waters)")
print("- ndci: Normalized Difference Chlorophyll Index")
print("- log_nir_red: Log10-transformed NIR/Red ratio (for calibration)")
print("- composite_day: Day of year for 8-day composite center")
print("- chlor_a: Chlorophyll-a concentration (if calibration coefficients provided)")
print("\nNote: Data are 8-day composites, providing cloud-free observations")
print("at 4x higher spatial resolution than 500m daily products.")

Detroit Aqua 250m 8-day features = 0
Exported: Detroit_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m.csv
Completed processing for Detroit Lake

UpperKlamath Aqua 250m 8-day features = 0
Exported: UKL_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m.csv
Completed processing for UpperKlamath Lake

Processing complete!

Output files contain the following indices at 250m resolution:
- nir_red_ratio: NIR/Red ratio (primary algorithm for turbid waters)
- ndci: Normalized Difference Chlorophyll Index
- log_nir_red: Log10-transformed NIR/Red ratio (for calibration)
- composite_day: Day of year for 8-day composite center
- chlor_a: Chlorophyll-a concentration (if calibration coefficients provided)

Note: Data are 8-day composites, providing cloud-free observations
at 4x higher spatial resolution than 500m daily products.
